In [18]:
import pandas as pd
import numpy as np

original_data_path = "../data/raw/"
data_binary_representation ="../data/processed/"

In [19]:
df = pd.read_csv(original_data_path + "wells.csv")
df_concentration = pd.read_csv(original_data_path + "concentration.csv")
df_head = pd.read_csv(original_data_path + "head.csv")
df_cores = pd.read_csv(original_data_path + "cores.csv")
df_pumping = pd.read_csv(original_data_path + "pumping.csv")

Сначала удалим все nan значение, затем удалим каскадно из других таблиц, после удалим дубликаты с округлением по x и y до целых и восстановим z.

In [20]:
df.loc[df['x'] == -999.0, "x"] = np.nan

nan_values_mask = df[['x','y','z']].isna().any(axis=1)

df_deleted = df[nan_values_mask]
df_deleted = df_deleted["well_id"].unique()
df = df[~nan_values_mask]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)

duplicated_mask = df.duplicated(subset=["round_x", "round_y", "z"], keep="first")
duplicated_idx = df[duplicated_mask]["well_id"].unique()

df = df[~duplicated_mask]
df = df.drop(columns=["round_x", "round_y"])

In [21]:
df_concentration = df_concentration[~df_concentration["well_id"].isin(df_deleted)]
df_head = df_head[~df_head["well_id"].isin(df_deleted)]
df_cores = df_cores[~df_cores["well_id"].isin(df_deleted)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(df_deleted)]

df_concentration = df_concentration[~df_concentration["well_id"].isin(duplicated_idx)]
df_head = df_head[~df_head["well_id"].isin(duplicated_idx)]
df_cores = df_cores[~df_cores["well_id"].isin(duplicated_idx)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(duplicated_idx)]

In [22]:
df.loc[(df["z"] > 5) & (df["z"] < 150), "z"] = -df["z"]

In [23]:
df.to_parquet(f"{data_binary_representation}wells.parquet", engine='pyarrow')

In [24]:
df_cores.loc[df_cores["z_top"] > 0, "z_top"] = -df_cores["z_top"]

df_cores = df_cores.sort_values(by=["well_id", "z_top"], ascending=[True, False])
df_cores["z_bottom"] = df_cores["z_bottom"].fillna(df_cores.groupby("well_id")["z_top"].shift(-1))

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom", "lithology"], na_position="last")
df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom"], keep="first")

df_cores.loc[df_cores["lithology"].isin(["Clay", "Caly"]), "lithology"] = "clay"
df_cores.loc[df_cores["lithology"].isin(["Gravel"]), "lithology"] = "gravel"
df_cores.loc[df_cores["lithology"].isin(["Loam"]), "lithology"] = "loam"
df_cores.loc[df_cores["lithology"].isin(["Sand", "SAND", "sand "]), "lithology"] = "sand"

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_max_deph = df_cores.groupby("well_id")["z_bottom"].cummin().shift(1)
same_well = df_cores["well_id"] == df_cores["well_id"].shift(1)
is_inner = same_well & (df_cores["z_bottom"] >= prev_max_deph)
df_cores = df_cores[~is_inner]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)
duplicated_wells = df.duplicated(subset=["round_x", "round_y"], keep=False)
wells_names = df[duplicated_wells].copy()

depths = df_cores.groupby("well_id")["z_bottom"].min().reset_index()
merged_df = wells_names.merge(depths, on="well_id")
trust_index = merged_df.groupby(by=["round_x", "round_y"])["z_bottom"].idxmin()
trusted_wells = merged_df.loc[trust_index, "well_id"]

bad_wells = set(wells_names["well_id"]) - set(trusted_wells)
df_cores = df_cores[~df_cores["well_id"].isin(bad_wells)]

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_z_bottom = df_cores.groupby("well_id")["z_bottom"].shift(1)
df_cores["z_top"] = np.minimum(df_cores["z_top"], prev_z_bottom.fillna(df_cores["z_top"]))

df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom", "lithology"], keep="first")

df_cores = df_cores.dropna(subset=["z_bottom", "lithology"])

In [25]:
df_cores.to_parquet(f"{data_binary_representation}cores.parquet")

In [26]:
df_head = df_head[df_head["head"] >= 0]

df_head = df_head.dropna(subset=["t", "head"])
df_head = df_head.drop_duplicates(subset=["well_id", "t"], keep="first")

df_head["t_datetime"] = pd.to_datetime(df_head["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_head["t_datetime"].isna()) |
    (df_head["t_datetime"] < "2020-01-01") |
    (df_head["t_datetime"] > "2025-10-31")
)
df_head = df_head[~bad_values]

df_head = df_head.drop(columns=["t"])
df_head = df_head.rename(columns={"t_datetime": "t"})

In [27]:
df_head.to_parquet(f"{data_binary_representation}head.parquet")

In [28]:
df_pumping.loc[df_pumping["units"] == "l/s", "pumping_rate"] = (df_pumping["pumping_rate"] * 24 * 60 * 60 / 1000)
df_pumping.loc[df_pumping["units"] == "l/s", "units"] = "m3/day"

df_pumping = df_pumping.dropna(subset=["pumping_rate"])

df_pumping["t_datetime"] = pd.to_datetime(df_pumping["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_pumping["t_datetime"].isna()) |
    (df_pumping["t_datetime"] < "2020-01-01") |
    (df_pumping["t_datetime"] > "2025-12-01")
)
df_pumping = df_pumping[~bad_values]

df_pumping = df_pumping.drop(columns=["t"])
df_pumping = df_pumping.rename(columns={"t_datetime":"t"})

df_pumping.loc[df_pumping["pumping_rate"] < 0, "pumping_rate"] = -df_pumping["pumping_rate"]

In [29]:
df_pumping.to_parquet(f"{data_binary_representation}pumping.parquet", engine="pyarrow")

In [30]:
df_concentration

,well_id,t,concentration
0,Well_0416,13.01.2022,0.21
2,Well_0110,02.02.2025,0.31
4,Well_0150,27.07.2024,1.74
6,Well_0570,18.10.2025,21.42
7,Well_0149,26.06.2021,1.8
...,...,...,...
2840,Well_0637,09.01.2020,0.03
2841,Well_0765,2023-08-08,ND
2842,Well_0118,02.05.2023,25.48
2843,Well_0681,23.06.2025,0.5


In [33]:
df_concentration.loc[df_concentration["concentration"].isin(["ND", "<0.05"]), "concentration"] = 0.0

df_concentration["concentration"] = pd.to_numeric(df_concentration["concentration"], errors="coerce")

df_concentration["t"] = pd.to_datetime(df_concentration["t"], format="%d.%m.%Y", errors="coerce")

df_concentration = df_concentration[df_concentration["concentration"] >= 0]

In [34]:
df_concentration.to_parquet(f"{data_binary_representation}concentration.parquet", engine="pyarrow")

In [35]:
df_abstraction = pd.read_csv(f"{original_data_path}abstraction.csv")
df_abstraction

,intake_name,x_center,y_center,t,total_abstraction_m3_day
0,Заречный водозабор,4000.0,3500.0,01.01.2020,26714.6
1,Заречный водозабор,4000.0,3500.0,01.02.2020,31722.3
2,Заречный водозабор,4000.0,3500.0,01.03.2020,38836.4
3,Заречный водозабор,4000.0,3500.0,01.04.2020,41624.0
4,Заречный водозабор,4000.0,3500.0,01.05.2020,42837.7
...,...,...,...,...,...
67,Заречный водозабор,4000.0,3500.0,01.08.2025,40293.2
68,Заречный водозабор,4000.0,3500.0,01.09.2025,35794.1
69,Заречный водозабор,4000.0,3500.0,01.10.2025,29547.1
70,Заречный водозабор,4000.0,3500.0,01.11.2025,28858.6


In [36]:
df_abstraction = df_abstraction.dropna(subset=["total_abstraction_m3_day"])

df_abstraction["t_datetime"] = pd.to_datetime(df_abstraction["t"], format="%d.%m.%Y", errors="coerce")

bad_values = (
    (df_abstraction["t_datetime"].isna()) |
    (df_abstraction["t_datetime"] < "2020-01-01") |
    (df_abstraction["t_datetime"] > "2025-12-01")
)

df_abstraction["t"] = pd.to_datetime(df_abstraction["t"], format="%d.%m.%Y", errors="coerce")

df_abstraction.drop(columns=["t_datetime"], inplace=True)

df_abstraction = df_abstraction[~bad_values]

In [37]:
df_abstraction.to_parquet(f"{data_binary_representation}abstraction.parquet", engine="pyarrow")

In [38]:
df_queries_test = pd.read_csv(f"{original_data_path}queries_test.csv")

In [39]:
df_queries_test.to_parquet(f"{data_binary_representation}queries_test.parquet", engine="pyarrow")

In [40]:
df_ground_truth_answers = pd.read_csv(f"{original_data_path}_ground_truth_answers.csv")

In [41]:
df_ground_truth_answers.to_parquet(f"{data_binary_representation}_ground_trust_answers.parquet", engine="pyarrow")

In [42]:
df_recharge = pd.read_csv(f"{original_data_path}recharge.csv")

In [43]:
df_recharge.dropna(subset=["recharge_mm"], inplace=True)
df_recharge["t"] = pd.to_datetime(df_recharge["t"], format="%d.%m.%Y", errors="coerce")
error_values = df_recharge["recharge_mm"] < 0

df_recharge = df_recharge[~error_values]

In [44]:
df_recharge.to_parquet(f"{data_binary_representation}recharge.parquet", engine="pyarrow")

In [46]:
df_sources = pd.read_csv(f"{original_data_path}sources.csv")

In [47]:
df_sources.to_parquet(f"{data_binary_representation}sources.parquet", engine="pyarrow")